# Prueba técnica Insights
## Ejercicio 4.1 – Investigación previa para agente ACH

**Santiago Osorio Gómez**  
**Cédula:** 1000.099.104  
**Fecha:** 12 de marzo de 2026

## Objetivo y criterio de presentación

Antes de implementar el agente, se realizó una investigación previa sobre el fondeo de cuentas de inversión vía ACH en Estados Unidos.

La investigación se organiza en cuatro bloques:

1. funcionamiento del flujo ACH;
2. requisitos para configurar un ACH pull y causas frecuentes de rechazo;
3. lógica de los routing numbers (ABA) y construcción de una tabla de lookup;
4. comparación entre ACH, wire y debit card para decidir cuál método conviene priorizar en Insights.

El objetivo no es construir una enciclopedia operativa, sino una síntesis profesional y suficiente para justificar el diseño del bot.  
Por eso, el entregable combina:

- una síntesis escrita en lenguaje propio;
- tablas operativas de apoyo;
- un seed lookup de routing numbers;
- y un apéndice de fuentes.

La información obtenida se utilizará directamente para el diseño del flujo conversacional y la implementación del prototipo CLI en Python.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# ============================================================
# EJERCICIO 4.1
# INVESTIGACION PREVIA PARA AGENTE ACH
# ============================================================

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    required_dirs = {"data", "docs", "notebooks", "outputs", "src"}

    for candidate in candidates:
        if candidate.exists():
            dir_names = {p.name for p in candidate.iterdir() if p.is_dir()}
            if required_dirs.issubset(dir_names):
                return candidate

    raise FileNotFoundError(
        "No se encontro la raiz del proyecto. "
        "Ejecuta este notebook dentro de ~/Documents/prueba_insights"
    )

ROOT = find_project_root()
DOCS_DIR = ROOT / "docs"
OUTPUTS_DIR = ROOT / "outputs"
EJ4_OUTPUT_DIR = OUTPUTS_DIR / "ejercicio_4"

DOCS_DIR.mkdir(parents=True, exist_ok=True)
EJ4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 200)

# ============================================================
# 1) CONTEXTO DE INSIGHTS
# ============================================================

company_context_df = pd.DataFrame([
    {
        "dimension": "Qué es Insights",
        "summary": "Plataforma/app de inversión para abrir cuentas de inversión en EE.UU. con foco en clientes de América Latina.",
        "why_it_matters_for_bot": "El bot debe hablar como una plataforma de inversión regulada, no como un banco comercial."
    },
    {
        "dimension": "Regulación y posicionamiento",
        "summary": "Insights comunica regulación SEC, FINRA y protección SIPC en sus materiales públicos.",
        "why_it_matters_for_bot": "El agente debe mantener tono sobrio, compliance-minded y evitar promesas excesivas."
    },
    {
        "dimension": "Timing ACH comunicado por Insights",
        "summary": "Insights comunica que un depósito vía ACH/transferencia puede reflejarse en 1-5 días hábiles.",
        "why_it_matters_for_bot": "El bot debe fijar expectativas de disponibilidad y no prometer acreditación inmediata."
    },
    {
        "dimension": "Restricciones operativas relevantes",
        "summary": "Para retiros, Insights indica verificación de cuenta a nombre del titular y primer retiro con validación bancaria.",
        "why_it_matters_for_bot": "El bot debe confirmar ownership, nombre del titular y canal de escalamiento si hay inconsistencia."
    },
])

# ============================================================
# 2) 4.1.1 - FLUJO ACH
# ============================================================

ach_flow_df = pd.DataFrame([
    {
        "step": 1,
        "actor": "Originator",
        "role": "Inicia la instrucción de débito o crédito ACH.",
        "agent_implication": "El cliente expresa intención de fondear la cuenta de inversión."
    },
    {
        "step": 2,
        "actor": "ODFI",
        "role": "El banco originador recibe la instrucción, la valida y la envía al operador ACH.",
        "agent_implication": "Insights o su partner de pagos necesita datos completos y autorización válida."
    },
    {
        "step": 3,
        "actor": "ACH Network / Operator",
        "role": "Clasifica, enruta y compensa el archivo ACH entre instituciones.",
        "agent_implication": "El bot debe explicar que ACH no es tiempo real; opera por ventanas/lotes."
    },
    {
        "step": 4,
        "actor": "RDFI",
        "role": "El banco receptor recibe la instrucción y decide posteo/aceptación/retorno.",
        "agent_implication": "Aquí se materializan retornos como R01, R02 o R03."
    },
    {
        "step": 5,
        "actor": "Receiver",
        "role": "Titular final de la cuenta bancaria o cuenta de inversión que recibe el efecto económico.",
        "agent_implication": "El cliente necesita saber cuándo el dinero queda disponible para operar."
    },
])

ach_timing_df = pd.DataFrame([
    {
        "mode": "ACH estándar",
        "network_timing": "La mayoría de pagos ACH liquida en 1 día bancario o menos; créditos pueden programarse same-day, next-day o en 2 días bancarios según reglas.",
        "when_to_recommend": "Opción base para fondeo regular, cuando el cliente no necesita urgencia intradía.",
        "insights_fit": "Consistente con la expectativa pública de 1-5 días hábiles comunicada por Insights."
    },
    {
        "mode": "Same-Day ACH",
        "network_timing": "Ventanas con liquidación aproximada a 1:00 p.m., 5:00 p.m. y 6:00 p.m. ET; límite actual de USD 1 millón por pago elegible.",
        "when_to_recommend": "Solo si el cliente necesita fondeo el mismo día hábil, el monto es elegible y el partner bancario soporta el flujo.",
        "insights_fit": "Debe tratarse como camino opcional, no como promesa base del bot."
    },
])

# ============================================================
# 3) 4.1.2 - REQUISITOS PARA ACH PULL
# ============================================================

ach_requirements_df = pd.DataFrame([
    {
        "item": "Bank name",
        "required_for": "Identificar institución y preparar el lookup del routing.",
        "what_bot_should_do": "Preguntar primero por banco."
    },
    {
        "item": "State",
        "required_for": "Elegir routing correcto cuando el banco usa rutas por estado/región.",
        "what_bot_should_do": "Preguntar estado antes de dar instrucciones."
    },
    {
        "item": "ABA routing number",
        "required_for": "Enrutar correctamente la instrucción ACH.",
        "what_bot_should_do": "Inferirlo desde banco+estado y pedir confirmación cuando corresponda."
    },
    {
        "item": "Bank account number",
        "required_for": "Identificar la cuenta origen del ACH pull.",
        "what_bot_should_do": "Solicitarlo después de explicar el proceso."
    },
    {
        "item": "Account type",
        "required_for": "Determinar checking vs savings.",
        "what_bot_should_do": "Pedirlo explícitamente."
    },
    {
        "item": "Legal name on bank account",
        "required_for": "Validación de titularidad/ownership.",
        "what_bot_should_do": "Confirmar que coincide con la cuenta del cliente en Insights."
    },
    {
        "item": "Authorization / consent",
        "required_for": "Cumplimiento operativo y de Nacha para débito ACH.",
        "what_bot_should_do": "Cerrar el flujo con una confirmación explícita."
    },
    {
        "item": "Verification method",
        "required_for": "Confirmar ownership y reducir fraude/errores.",
        "what_bot_should_do": "Explicar que puede hacerse por instant auth o micro-deposits."
    },
    {
        "item": "Funding limits",
        "required_for": "Evitar expectativas erróneas sobre montos.",
        "what_bot_should_do": "Explicar que los límites dependen del proveedor/riesgo; usar límites mock/documentados en el prototipo."
    },
    {
        "item": "Funds availability timing",
        "required_for": "Gestionar expectativa del cliente.",
        "what_bot_should_do": "Informar que la acreditación no es necesariamente inmediata."
    },
])

verification_methods_df = pd.DataFrame([
    {
        "method": "Instant bank verification",
        "description": "Conexión autenticada para recuperar y verificar routing/account/token + eventualmente ownership/balance.",
        "pros": "Menos errores manuales y menor fricción.",
        "cons": "Depende de cobertura del proveedor.",
    },
    {
        "method": "Micro-deposits",
        "description": "Pequeño depósito(s) que el cliente confirma para probar control de la cuenta.",
        "pros": "Funciona como fallback cuando no hay instant auth.",
        "cons": "Más lento: puede tomar 1-2 días hábiles o más según el flujo."
    },
])

funding_limits_df = pd.DataFrame([
    {
        "method": "Same-Day ACH",
        "typical_limit": "Hasta USD 1,000,000 por transacción elegible a nivel de red.",
        "comment": "No implica que todas las plataformas ofrezcan ese límite al cliente final."
    },
    {
        "method": "Originated ACH (ejemplo broker/fintech)",
        "typical_limit": "Límites definidos por proveedor y perfil de riesgo; ejemplo público: Robinhood muestra límites diarios mucho mayores para ACH que para tarjeta.",
        "comment": "En el prototipo conviene usar límites mock configurables, no asumir el límite real de Insights."
    },
    {
        "method": "Insights ACH availability",
        "typical_limit": "La ayuda pública de Insights no publica un límite ACH universal en el material consultado.",
        "comment": "Sí publica ventana de disponibilidad de 1-5 días hábiles para ver reflejado el depósito."
    },
])

nacha_return_codes_df = pd.DataFrame([
    {
        "code": "R01",
        "meaning": "Insufficient Funds",
        "plain_english": "La cuenta existe, pero no tenía fondos disponibles suficientes al momento del débito.",
        "message_insights_style": "No pudimos completar tu fondeo ACH porque tu cuenta bancaria no tenía fondos suficientes. Intenta nuevamente con saldo disponible o usa otro método de fondeo."
    },
    {
        "code": "R02",
        "meaning": "Account Closed",
        "plain_english": "La cuenta estaba cerrada al momento del intento.",
        "message_insights_style": "No pudimos completar tu fondeo ACH porque la cuenta bancaria reportada aparece cerrada. Verifica tus datos o registra otra cuenta bancaria."
    },
    {
        "code": "R03",
        "meaning": "No Account / Unable to Locate Account",
        "plain_english": "La cuenta no pudo localizarse o no coincide con la información del registro.",
        "message_insights_style": "No pudimos localizar la cuenta bancaria con la información proporcionada. Revisa routing number, account number y nombre del titular antes de intentar nuevamente."
    },
])

# ============================================================
# 4) 4.1.3 - ABA ROUTING NUMBERS
# Seed lookup para prototipo rapido, no directorio nacional exhaustivo.
# ============================================================

aba_explanation_df = pd.DataFrame([
    {
        "topic": "Qué es",
        "summary": "Un ABA routing number es un identificador de 9 dígitos para enrutar pagos en EE.UU."
    },
    {
        "topic": "Estructura",
        "summary": "Se compone de Federal Reserve Routing Symbol, ABA Institution Identifier y Check Digit."
    },
    {
        "topic": "Por qué cambia por estado",
        "summary": "Bancos grandes pueden usar rutas diferentes por estado/región, por banco heredado o por cómo/ dónde se abrió la cuenta."
    },
    {
        "topic": "Regla para el agente",
        "summary": "Preguntar banco + estado primero; luego hacer lookup; si hay ambigüedad, pedir confirmación contra cheque o app bancaria."
    },
])

routing_lookup_df = pd.DataFrame([
    {
        "bank": "Bank of America",
        "state_or_region": "Texas",
        "routing_number": "111000025",
        "coverage_note": "Seed lookup para el prototipo.",
        "source_label": "Bank of America routing FAQ"
    },
    {
        "bank": "Chase",
        "state_or_region": "Texas",
        "routing_number": "111000614",
        "coverage_note": "Seed lookup para el prototipo.",
        "source_label": "Chase state list (publicly reported from Chase source)"
    },
    {
        "bank": "Wells Fargo",
        "state_or_region": "California",
        "routing_number": "121000248",
        "coverage_note": "La página oficial indica que en Southern California puede verse otro número válido en cheques.",
        "source_label": "Wells Fargo routing page"
    },
    {
        "bank": "Citibank",
        "state_or_region": "Texas",
        "routing_number": "113193532",
        "coverage_note": "Seed lookup para el prototipo.",
        "source_label": "Public routing directory / Citibank Texas listing"
    },
    {
        "bank": "Santander",
        "state_or_region": "NJ / NY / PA / DE / MD",
        "routing_number": "231372691",
        "coverage_note": "Ruta pública por grupo de estados.",
        "source_label": "Santander routing page"
    },
    {
        "bank": "TD Bank",
        "state_or_region": "Florida",
        "routing_number": "067014822",
        "coverage_note": "Ruta pública por estado.",
        "source_label": "TD direct deposit page"
    },
    {
        "bank": "U.S. Bank",
        "state_or_region": "California - North",
        "routing_number": "121122676",
        "coverage_note": "Seed lookup para el prototipo.",
        "source_label": "U.S. Bank routing directory"
    },
    {
        "bank": "Regions",
        "state_or_region": "Texas",
        "routing_number": "111900785",
        "coverage_note": "Tomado de material público de switch-kit; conviene confirmación en app/check.",
        "source_label": "Regions switch kit"
    },
    {
        "bank": "Capital One 360",
        "state_or_region": "Nationwide / online",
        "routing_number": "031176110",
        "coverage_note": "Útil como caso de banco online con ruta única en el prototipo.",
        "source_label": "Capital One 360 public routing references"
    },
    {
        "bank": "Truist",
        "state_or_region": "General / new accounts",
        "routing_number": "061000104",
        "coverage_note": "Algunas cuentas heredadas de BB&T pueden seguir usando rutas legacy.",
        "source_label": "Public Truist routing guide"
    },
])

routing_decision_df = pd.DataFrame([
    {
        "user_input": "Bank of America en Texas",
        "agent_step": "Normalizar banco = Bank of America; estado = Texas.",
        "bot_action": "Buscar coincidencia exacta en routing_lookup.csv.",
        "expected_output": "Routing sugerido: 111000025."
    },
    {
        "user_input": "Bank of America en Texas",
        "agent_step": "Si el lookup es único, mostrar el routing y aclarar que puede confirmarlo en su cheque o app.",
        "bot_action": "Continuar con account type, account number y autorización.",
        "expected_output": "El bot no se detiene por no conocer el routing de memoria el cliente."
    },
])

# ============================================================
# 5) 4.1.4 - ACH vs Wire vs Debit Card
# ============================================================

payments_comparison_df = pd.DataFrame([
    {
        "method": "ACH",
        "speed": "Mismo día en casos elegibles; típicamente 1 día bancario o más; en Insights, el abono visible se comunica en 1-5 días hábiles.",
        "cost": "Generalmente bajo o gratis para el usuario final.",
        "typical_limits": "Más alto que tarjeta en muchos brokers/fintechs; depende del proveedor.",
        "reversibility": "Más reversible/retornable que wire bajo reglas ACH.",
        "user_experience": "Buena para fondeo recurrente; algo menos inmediata que tarjeta.",
        "recommended_for_insights": "Método preferido para fondeo recurrente y costo bajo."
    },
    {
        "method": "Wire",
        "speed": "Usualmente mismo día hábil si se envía antes del cutoff.",
        "cost": "Más costoso que ACH.",
        "typical_limits": "Suele admitir montos altos.",
        "reversibility": "Muy difícil de revertir una vez enviado.",
        "user_experience": "Más fricción, pero útil para urgencia y alto valor.",
        "recommended_for_insights": "Conviene para urgencia o montos altos."
    },
    {
        "method": "Debit Card",
        "speed": "Puede ser casi inmediato.",
        "cost": "Suele ser más costoso para el originador y/o con límites más bajos.",
        "typical_limits": "En brokers/fintechs, normalmente mucho más bajo que ACH.",
        "reversibility": "Intermedio; depende del riel y del proveedor.",
        "user_experience": "Muy simple para montos pequeños y experiencia rápida.",
        "recommended_for_insights": "Útil como opción secundaria para convenience/small ticket, no como método principal."
    },
])

# ============================================================
# 6) FUENTES
# ============================================================

sources_df = pd.DataFrame([
    {"topic": "Insights overview", "source_label": "Insights homepage", "url": "https://www.insightswm.com/"},
    {"topic": "Insights help center / what is Insights", "source_label": "Insights help center", "url": "https://www.insightswm.com/help-center"},
    {"topic": "Insights ACH deposit timing", "source_label": "Insights help article - no veo abonado", "url": "https://help.insightswm.com/hc/es-419/articles/10572148616091-No-veo-abonado-el-dinero-en-mi-cuenta-de-inversi%C3%B3n-de-Insights"},
    {"topic": "Insights ACH withdrawal timing", "source_label": "Insights help article - retiros", "url": "https://help.insightswm.com/hc/es-419/articles/9068243168283--C%C3%B3mo-realizar-un-retiro-en-Insights"},
    {"topic": "ACH flow", "source_label": "Nacha - How ACH Payments Work", "url": "https://www.nacha.org/content/how-ach-payments-work"},
    {"topic": "ACH speed and facts", "source_label": "Nacha - ACH Payments Fact Sheet", "url": "https://www.nacha.org/content/ach-payments-fact-sheet"},
    {"topic": "Same-Day ACH windows", "source_label": "Federal Reserve - FedACH Processing Schedule", "url": "https://www.frbservices.org/resources/resource-centers/same-day-ach/fedach-processing-schedule.html"},
    {"topic": "Same-Day ACH eligibility", "source_label": "Federal Reserve - FedACH SameDay Service", "url": "https://www.frbservices.org/financial-services/ach/same-day-service.html"},
    {"topic": "Same-Day ACH limit", "source_label": "Nacha - $1M limit", "url": "https://www.nacha.org/million"},
    {"topic": "ABA routing structure", "source_label": "ABA Routing Number", "url": "https://www.aba.com/about-us/routing-number"},
    {"topic": "ABA routing structure detail", "source_label": "ABA Routing Number Policy", "url": "https://www.aba.com/news-research/analysis-guides/routing-number-policy-procedures"},
    {"topic": "Instant verification / balances / ownership", "source_label": "Stripe Financial Connections", "url": "https://stripe.com/us/financial-connections"},
    {"topic": "ACH direct debit with verification", "source_label": "Stripe docs", "url": "https://docs.stripe.com/financial-connections/ach-direct-debit-payments"},
    {"topic": "Same-day micro-deposits", "source_label": "Plaid docs", "url": "https://plaid.com/docs/auth/coverage/same-day/"},
    {"topic": "Automated micro-deposits", "source_label": "Plaid docs automated", "url": "https://plaid.com/docs/auth/coverage/automated/"},
    {"topic": "ACH return codes", "source_label": "PayJunction return codes", "url": "https://support.payjunction.com/hc/en-us/articles/218502067-ACH-Return-Codes-and-Definitions"},
    {"topic": "ACH return codes alternative", "source_label": "AndDone return codes", "url": "https://help.anddone.com/ach-return-codes-explained"},
    {"topic": "BoA routing", "source_label": "Bank of America routing FAQ", "url": "https://www.bankofamerica.com/deposits/routing-number-faqs/"},
    {"topic": "Chase routing", "source_label": "Bankrate state list citing Chase", "url": "https://www.bankrate.com/banking/checking/what-is-a-routing-number/"},
    {"topic": "Wells routing", "source_label": "Wells Fargo routing page", "url": "https://www.wellsfargo.com/help/routing-number/"},
    {"topic": "Santander routing", "source_label": "Santander routing page", "url": "https://www.santanderbank.com/personal/banking/checking-resources/routing-number"},
    {"topic": "TD routing", "source_label": "TD direct deposit page", "url": "https://www.td.com/us/en/personal-banking/direct-deposit"},
    {"topic": "U.S. Bank routing", "source_label": "U.S. Bank routing directory", "url": "https://www.usbank.com/es/bank-accounts/checking-accounts/checking-customer-resources/aba-routing-number.html"},
    {"topic": "Regions routing", "source_label": "Regions switch kit", "url": "https://www.regions.com/virtualdocuments/Business_SwitchKit_Forms.pdf"},
    {"topic": "Citibank routing", "source_label": "Citibank public routing guide", "url": "https://www.getnickel.com/bank-routing-numbers/citibank"},
    {"topic": "Capital One routing", "source_label": "Capital One help center", "url": "https://www.capitalone.com/help-center/checking-savings/routing-account-numbers/"},
    {"topic": "Capital One routing reference", "source_label": "Capital One 360 public routing reference", "url": "https://www.hustlermoneyblog.com/capital-one-360-money-market-review/"},
    {"topic": "Truist routing", "source_label": "Truist public routing guide", "url": "https://www.getnickel.com/bank-routing-numbers/truist"},
    {"topic": "Wire reversibility", "source_label": "OCC / HelpWithMyBank", "url": "https://www2.helpwithmybank.gov/help-topics/fraud-scams/scams/wire-transfer-scams/wire-transfer-scams-held-liable.html"},
    {"topic": "Funding limits example", "source_label": "Robinhood transfer limits", "url": "https://robinhood.com/us/en/support/articles/spending-limits/"},
])

# ============================================================
# 7) SINTESIS EJECUTIVA PARA EXPORTAR
# ============================================================

research_markdown = """# Ejercicio 4.1 - Investigación previa
**Prueba técnica Insights**  
**Santiago Osorio Gómez**

## 4.1.1 ¿Cómo funciona ACH en EE.UU.?
El flujo ACH se entiende como una cadena de cinco actores: **Originator -> ODFI -> ACH Network / Operator -> RDFI -> Receiver**.  
El Originator inicia la instrucción; el ODFI la valida y la origina; el operador ACH la compensa y enruta; el RDFI la recibe, la postea o la retorna; y el Receiver recibe el abono o sufre el débito.

ACH no es un riel en tiempo real: funciona por archivos y ventanas de procesamiento.  
Como criterio práctico para el agente, la opción base debe ser **ACH estándar**, mientras que **Same-Day ACH** solo debería recomendarse cuando el cliente necesita fondeo urgente el mismo día hábil, el monto es elegible y la institución participante soporta ese flujo.

## 4.1.2 Requisitos para fondear una cuenta de inversión vía ACH
Para configurar un **ACH pull**, el usuario necesita como mínimo:
- nombre del banco;
- estado donde abrió la cuenta;
- ABA routing number;
- account number;
- tipo de cuenta (checking o savings);
- nombre legal del titular;
- autorización expresa del débito ACH.

Adicionalmente, el proceso suele requerir **verificación de titularidad** por instant verification o micro-deposits, y en algunos casos verificación de balances.

Los códigos de retorno más relevantes para el prototipo son:
- **R01**: fondos insuficientes;
- **R02**: cuenta cerrada;
- **R03**: cuenta inexistente o no localizable.

## 4.1.3 Routing numbers (ABA): lógica y lookup por banco y estado
Un **ABA routing number** es un identificador de 9 dígitos para pagos en EE.UU.  
No es suficiente con preguntar el banco: los bancos grandes pueden tener múltiples routing numbers por estado, región o legado operativo. Por eso, el agente debe cumplir esta regla:

1. preguntar banco;
2. preguntar estado;
3. hacer lookup;
4. mostrar el routing sugerido;
5. pedir confirmación si hay ambigüedad.

Para este prototipo se construye una tabla curada de 10 bancos frecuentes, suficiente como **seed lookup** y no como directorio nacional exhaustivo.

## 4.1.4 Comparativa: ACH vs Wire vs Debit Card
La conclusión operativa es:

- **ACH** debe ser el método preferido para Insights por su bajo costo, buena experiencia y adecuación al fondeo recurrente.
- **Wire** debe recomendarse cuando el cliente prioriza urgencia o monto alto y acepta mayor costo.
- **Debit Card** sirve mejor como opción complementaria para conveniencia e importes pequeños, no como riel principal del bot.

## Conclusión de 4.1
La investigación sugiere que el agente de Insights debe ser:
- **compliance-minded**;
- explícito sobre tiempos de disponibilidad;
- estricto con la recolección de banco + estado antes de cualquier instrucción;
- y prudente al comunicar routing numbers, tratándolos como inferencia asistida con confirmación cuando sea necesario.

Esta síntesis deja lista la base conceptual para diseñar en 4.2 un bot CLI simple, determinístico y modular.
"""

# ============================================================
# 8) EXPORTACION
# ============================================================

company_context_df.to_csv(EJ4_OUTPUT_DIR / "ejercicio_4_1_company_context.csv", index=False)
ach_flow_df.to_csv(EJ4_OUTPUT_DIR / "ejercicio_4_1_ach_flow.csv", index=False)
ach_timing_df.to_csv(EJ4_OUTPUT_DIR / "ejercicio_4_1_ach_timing.csv", index=False)
ach_requirements_df.to_csv(EJ4_OUTPUT_DIR / "ejercicio_4_1_ach_requirements.csv", index=False)
verification_methods_df.to_csv(EJ4_OUTPUT_DIR / "ejercicio_4_1_verification_methods.csv", index=False)
funding_limits_df.to_csv(EJ4_OUTPUT_DIR / "ejercicio_4_1_funding_limits.csv", index=False)
nacha_return_codes_df.to_csv(EJ4_OUTPUT_DIR / "ejercicio_4_1_return_codes.csv", index=False)
aba_explanation_df.to_csv(EJ4_OUTPUT_DIR / "ejercicio_4_1_aba_explanation.csv", index=False)
routing_lookup_df.to_csv(EJ4_OUTPUT_DIR / "ejercicio_4_1_routing_lookup_seed.csv", index=False)
routing_decision_df.to_csv(EJ4_OUTPUT_DIR / "ejercicio_4_1_routing_decision_logic.csv", index=False)
payments_comparison_df.to_csv(EJ4_OUTPUT_DIR / "ejercicio_4_1_payments_comparison.csv", index=False)
sources_df.to_csv(EJ4_OUTPUT_DIR / "ejercicio_4_1_sources.csv", index=False)

(DOCS_DIR / "ejercicio_4_1_investigacion_previa.md").write_text(research_markdown, encoding="utf-8")
(EJ4_OUTPUT_DIR / "ejercicio_4_1_investigacion_previa.txt").write_text(research_markdown, encoding="utf-8")

# ============================================================
# 9) PRESENTACION EN NOTEBOOK
# ============================================================

display(Markdown("# Ejercicio 4.1 - Investigación previa"))

display(Markdown("## 1. Contexto de Insights"))
display(company_context_df)

display(Markdown("## 2. Flujo ACH y tiempos"))
display(ach_flow_df)
display(ach_timing_df)

display(Markdown("## 3. Requisitos para fondear una cuenta de inversión vía ACH"))
display(ach_requirements_df)
display(verification_methods_df)
display(funding_limits_df)
display(nacha_return_codes_df)

display(Markdown("## 4. ABA routing numbers y lógica de lookup"))
display(aba_explanation_df)
display(routing_lookup_df)
display(routing_decision_df)

display(Markdown("## 5. Comparativa: ACH vs Wire vs Debit Card"))
display(payments_comparison_df)

display(Markdown("## 6. Síntesis ejecutiva"))
display(Markdown(research_markdown))

display(Markdown("## 7. Fuentes utilizadas"))
display(sources_df)

print("Archivos generados:")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_1_company_context.csv'}")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_1_ach_flow.csv'}")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_1_ach_timing.csv'}")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_1_ach_requirements.csv'}")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_1_verification_methods.csv'}")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_1_funding_limits.csv'}")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_1_return_codes.csv'}")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_1_aba_explanation.csv'}")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_1_routing_lookup_seed.csv'}")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_1_routing_decision_logic.csv'}")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_1_payments_comparison.csv'}")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_1_sources.csv'}")
print(f"- {DOCS_DIR / 'ejercicio_4_1_investigacion_previa.md'}")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_1_investigacion_previa.txt'}")

# Ejercicio 4.1 - Investigación previa

## 1. Contexto de Insights

,dimension,summary,why_it_matters_for_bot
0,Qué es Insights,Plataforma/app de inversión para abrir cuentas de inversión en EE.UU. con foco en clientes de América Latina.,"El bot debe hablar como una plataforma de inversión regulada, no como un banco comercial."
1,Regulación y posicionamiento,"Insights comunica regulación SEC, FINRA y protección SIPC en sus materiales públicos.","El agente debe mantener tono sobrio, compliance-minded y evitar promesas excesivas."
2,Timing ACH comunicado por Insights,Insights comunica que un depósito vía ACH/transferencia puede reflejarse en 1-5 días hábiles.,El bot debe fijar expectativas de disponibilidad y no prometer acreditación inmediata.
3,Restricciones operativas relevantes,"Para retiros, Insights indica verificación de cuenta a nombre del titular y primer retiro con validación bancaria.","El bot debe confirmar ownership, nombre del titular y canal de escalamiento si hay inconsistencia."


## 2. Flujo ACH y tiempos

,step,actor,role,agent_implication
0,1,Originator,Inicia la instrucción de débito o crédito ACH.,El cliente expresa intención de fondear la cuenta de inversión.
1,2,ODFI,"El banco originador recibe la instrucción, la valida y la envía al operador ACH.",Insights o su partner de pagos necesita datos completos y autorización válida.
2,3,ACH Network / Operator,"Clasifica, enruta y compensa el archivo ACH entre instituciones.",El bot debe explicar que ACH no es tiempo real; opera por ventanas/lotes.
3,4,RDFI,El banco receptor recibe la instrucción y decide posteo/aceptación/retorno.,"Aquí se materializan retornos como R01, R02 o R03."
4,5,Receiver,Titular final de la cuenta bancaria o cuenta de inversión que recibe el efecto económico.,El cliente necesita saber cuándo el dinero queda disponible para operar.


...

## 6. Síntesis ejecutiva

# Ejercicio 4.1 - Investigación previa
**Prueba técnica Insights**  
**Santiago Osorio Gómez**

## 4.1.1 ¿Cómo funciona ACH en EE.UU.?
El flujo ACH se entiende como una cadena de cinco actores: **Originator -> ODFI -> ACH Network / Operator -> RDFI -> Receiver**.  
El Originator inicia la instrucción; el ODFI la valida y la origina; el operador ACH la compensa y enruta; el RDFI la recibe, la postea o la retorna; y el Receiver recibe el abono o sufre el débito.

ACH no es un riel en tiempo real: funciona por archivos y ventanas de procesamiento.  
Como criterio práctico para el agente, la opción base debe ser **ACH estándar**, mientras que **Same-Day ACH** solo debería recomendarse cuando el cliente necesita fondeo urgente el mismo día hábil, el monto es elegible y la institución participante soporta ese flujo.

## 4.1.2 Requisitos para fondear una cuenta de inversión vía ACH
Para configurar un **ACH pull**, el usuario necesita como mínimo:
- nombre del banco;
- estado donde abrió la cuenta;
- ABA routing number;
- account number;
- tipo de cuenta (checking o savings);
- nombre legal del titular;
- autorización expresa del débito ACH.

Adicionalmente, el proceso suele requerir **verificación de titularidad** por instant verification o micro-deposits, y en algunos casos verificación de balances.

Los códigos de retorno más relevantes para el prototipo son:
- **R01**: fondos insuficientes;
- **R02**: cuenta cerrada;
- **R03**: cuenta inexistente o no localizable.

## 4.1.3 Routing numbers (ABA): lógica y lookup por banco y estado
Un **ABA routing number** es un identificador de 9 dígitos para pagos en EE.UU.  
No es suficiente con preguntar el banco: los bancos grandes pueden tener múltiples routing numbers por estado, región o legado operativo. Por eso, el agente debe cumplir esta regla:

1. preguntar banco;
2. preguntar estado;
3. hacer lookup;
4. mostrar el routing sugerido;
5. pedir confirmación si hay ambigüedad.

Para este prototipo se construye una tabla curada de 10 bancos frecuentes, suficiente como **seed lookup** y no como directorio nacional exhaustivo.

## 4.1.4 Comparativa: ACH vs Wire vs Debit Card
La conclusión operativa es:

- **ACH** debe ser el método preferido para Insights por su bajo costo, buena experiencia y adecuación al fondeo recurrente.
- **Wire** debe recomendarse cuando el cliente prioriza urgencia o monto alto y acepta mayor costo.
- **Debit Card** sirve mejor como opción complementaria para conveniencia e importes pequeños, no como riel principal del bot.

## Conclusión de 4.1
La investigación sugiere que el agente de Insights debe ser:
- **compliance-minded**;
- explícito sobre tiempos de disponibilidad;
- estricto con la recolección de banco + estado antes de cualquier instrucción;
- y prudente al comunicar routing numbers, tratándolos como inferencia asistida con confirmación cuando sea necesario.

Esta síntesis deja lista la base conceptual para diseñar en 4.2 un bot CLI simple, determinístico y modular.


## 7. Fuentes utilizadas

,topic,source_label,url
0,Insights overview,Insights homepage,https://www.insightswm.com/
1,Insights help center / what is Insights,Insights help center,https://www.insightswm.com/help-center
2,Insights ACH deposit timing,Insights help article - no veo abonado,https://help.insightswm.com/hc/es-419/articles/10572148616091-No-veo-abonado-el-dinero-en-mi-cuenta-de-inversi%C3%B3n-de-Insights
3,Insights ACH withdrawal timing,Insights help article - retiros,https://help.insightswm.com/hc/es-419/articles/9068243168283--C%C3%B3mo-realizar-un-retiro-en-Insights
4,ACH flow,Nacha - How ACH Payments Work,https://www.nacha.org/content/how-ach-payments-work
5,ACH speed and facts,Nacha - ACH Payments Fact Sheet,https://www.nacha.org/content/ach-payments-fact-sheet
6,Same-Day ACH windows,Federal Reserve - FedACH Processing Schedule,https://www.frbservices.org/resources/resource-centers/same-day-ach/fedach-processing-schedule.html
7,Same-Day ACH eligibility,Federal Reserve - FedACH SameDay Service,https://www.frbservices.org/financial-services/ach/same-day-service.html
8,Same-Day ACH limit,Nacha - $1M limit,https://www.nacha.org/million
9,ABA routing structure,ABA Routing Number,https://www.aba.com/about-us/routing-number


Archivos generados:
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/ejercicio_4_1_company_context.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/ejercicio_4_1_ach_flow.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/ejercicio_4_1_ach_timing.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/ejercicio_4_1_ach_requirements.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/ejercicio_4_1_verification_methods.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/ejercicio_4_1_funding_limits.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/ejercicio_4_1_return_codes.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/ejercicio_4_1_aba_explanation.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/ejercicio_4_1_routing_lookup_seed.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/ejercicio_

## Cierre de 4.1

La investigación previa deja tres conclusiones prácticas para el diseño del agente:

1. **ACH debe ser el riel principal del bot**, porque es el método más razonable para fondeo recurrente de cuentas de inversión en términos de costo, experiencia y operación.
2. **Banco + estado debe ser la primera pregunta obligatoria**, porque la lógica del routing number depende de esa combinación y no debe inferirse a ciegas.
3. **La lógica crítica del agente debe ser determinística**, especialmente en:
   - recolección de datos,
   - inferencia/lookup de routing,
   - confirmación de datos,
   - y manejo de fallos como `R01` y `R03`.

Con esto queda lista la base conceptual para pasar a **4.2 Diseño del agente**, manteniendo una implementación simple y rápida.

## Ejercicio 4.2–4.4 – Diseño e implementación mínima del agente ACH

Para el desarrollo del chatbot se eligió una arquitectura deliberadamente simple y defendible, adecuada al tiempo disponible y al objetivo del ejercicio.

### Decisiones de diseño
- implementación en **Python**
- interfaz por **consola (CLI)**
- **máquina de estados** para controlar el flujo
- lógica **determinística** para la parte crítica
- capa separada de **lookup de routing number**
- capa separada de **simulación de outcomes**
- manejo de **historial conversacional**
- sin dependencia de LLM en la lógica principal

### Justificación
Este enfoque permite:
- construir un prototipo funcional en poco tiempo;
- probarlo fácilmente;
- depurar la lógica sin ambigüedades;
- y presentar un entregable profesional sin sobreingeniería.

### Casos cubiertos
El agente cubre los siguientes outcomes mínimos:
- `success`
- `R01`
- `R03`

### Flujo base
1. preguntar banco;
2. preguntar estado;
3. inferir routing number;
4. explicar el proceso de funding ACH;
5. recolectar datos necesarios;
6. confirmar antes de enviar;
7. simular outcome;
8. guardar transcript de la conversación.

In [2]:
from pathlib import Path

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    required_dirs = {"data", "docs", "notebooks", "outputs", "src"}

    for candidate in candidates:
        if candidate.exists():
            dir_names = {p.name for p in candidate.iterdir() if p.is_dir()}
            if required_dirs.issubset(dir_names):
                return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Ejecuta este notebook dentro de ~/Documents/prueba_insights"
    )

ROOT = find_project_root()
SRC_DIR = ROOT / "src"

ach_services_code_fixed = """
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Optional, Any

import pandas as pd

from ach_state import SessionState


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    required_dirs = {"data", "docs", "notebooks", "outputs", "src"}

    for candidate in candidates:
        if candidate.exists():
            dir_names = {p.name for p in candidate.iterdir() if p.is_dir()}
            if required_dirs.issubset(dir_names):
                return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Ejecuta el script dentro de ~/Documents/prueba_insights"
    )


def normalize_text(text: str) -> str:
    cleaned = "".join(ch.lower() if ch.isalnum() else " " for ch in str(text))
    return " ".join(cleaned.split())


def load_routing_lookup() -> pd.DataFrame:
    root = find_project_root()
    candidate_paths = [
        root / "data" / "routing_lookup_seed.csv",
        root / "outputs" / "ejercicio_4" / "ejercicio_4_1_routing_lookup_seed.csv",
    ]

    existing = [p for p in candidate_paths if p.exists()]
    if not existing:
        raise FileNotFoundError(
            "No encontré el archivo de routing lookup. "
            "Esperado en data/routing_lookup_seed.csv o "
            "outputs/ejercicio_4/ejercicio_4_1_routing_lookup_seed.csv"
        )

    # IMPORTANTE:
    # routing_number debe leerse como texto para no perder ceros a la izquierda
    routing_df = pd.read_csv(existing[0], dtype={"routing_number": "string"})
    routing_df["routing_number"] = (
        routing_df["routing_number"]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
        .str.zfill(9)
    )
    routing_df["bank_norm"] = routing_df["bank"].map(normalize_text)
    routing_df["region_norm"] = routing_df["state_or_region"].map(normalize_text)
    return routing_df


def _state_matches(user_state: str, row_region: str) -> bool:
    user_state_norm = normalize_text(user_state)
    row_region_norm = normalize_text(row_region)

    if user_state_norm == row_region_norm:
        return True

    if user_state_norm in row_region_norm:
        return True

    if "nationwide" in row_region_norm or "general" in row_region_norm or "online" in row_region_norm:
        return True

    return False


def lookup_routing(bank: str, state_name: str, routing_df: pd.DataFrame) -> Dict[str, Optional[str]]:
    bank_norm = normalize_text(bank)
    state_norm = normalize_text(state_name)

    exact_matches = routing_df[
        (routing_df["bank_norm"] == bank_norm)
        & (routing_df["region_norm"].map(lambda x: _state_matches(state_norm, x)))
    ].copy()

    if not exact_matches.empty:
        row = exact_matches.iloc[0]
        return {
            "found": True,
            "bank": str(row["bank"]),
            "state_or_region": str(row["state_or_region"]),
            "routing_number": str(row["routing_number"]),
            "match_note": "Coincidencia encontrada en seed lookup por banco y estado/región.",
        }

    partial_matches = routing_df[
        (routing_df["bank_norm"].str.contains(bank_norm, regex=False))
        & (routing_df["region_norm"].map(lambda x: _state_matches(state_norm, x)))
    ].copy()

    if not partial_matches.empty:
        row = partial_matches.iloc[0]
        return {
            "found": True,
            "bank": str(row["bank"]),
            "state_or_region": str(row["state_or_region"]),
            "routing_number": str(row["routing_number"]),
            "match_note": "Coincidencia parcial encontrada en seed lookup; conviene confirmación visual en app o cheque.",
        }

    return {
        "found": False,
        "bank": None,
        "state_or_region": None,
        "routing_number": None,
        "match_note": "No se encontró una coincidencia en el lookup seed.",
    }


def explain_ach(bank: str, state_name: str, routing_number: str) -> str:
    return (
        f"Perfecto. Para {bank} en {state_name}, el routing sugerido en el lookup seed es {routing_number}.\\n"
        "El proceso ACH funciona así: recopilamos tus datos bancarios, confirmamos la información, "
        "originamos la instrucción y el banco receptor puede aceptar o retornar la operación. "
        "En general, el fondeo ACH no es instantáneo y puede reflejarse en un rango de 1 a 5 días hábiles."
    )


def validate_account_type(value: str) -> Optional[str]:
    value_norm = normalize_text(value)
    if value_norm in {"checking", "corriente"}:
        return "checking"
    if value_norm in {"savings", "ahorros"}:
        return "savings"
    return None


def clean_account_number(value: str) -> Optional[str]:
    digits = "".join(ch for ch in str(value) if ch.isdigit())
    if len(digits) < 4:
        return None
    return digits


def validate_amount(value: str) -> Optional[float]:
    try:
        amount = float(str(value).replace(",", "").strip())
    except ValueError:
        return None

    if amount <= 0:
        return None
    return round(amount, 2)


def simulate_submission(scenario: str, session: SessionState) -> Dict[str, Optional[str]]:
    scenario = scenario.lower().strip()

    if scenario == "success":
        return {
            "status": "success",
            "code": None,
            "message": (
                f"Tu solicitud ACH fue creada correctamente por USD {session.amount:.2f}. "
                "La referencia quedó en estado submitted y el fondeo puede reflejarse en 1 a 5 días hábiles."
            ),
        }

    if scenario == "r01":
        return {
            "status": "returned",
            "code": "R01",
            "message": (
                "La operación fue retornada con código R01 (insufficient funds). "
                "La cuenta existe, pero no tenía fondos suficientes para completar el débito."
            ),
        }

    if scenario == "r03":
        return {
            "status": "returned",
            "code": "R03",
            "message": (
                "La operación fue retornada con código R03 (no account / unable to locate account). "
                "No fue posible localizar la cuenta con los datos proporcionados."
            ),
        }

    raise ValueError("Escenario no soportado. Usa: success, r01 o r03.")


def build_transcript_text(session: SessionState) -> str:
    lines = []
    lines.append("ACH BOT TRANSCRIPT")
    lines.append(f"Scenario: {session.scenario}")
    lines.append(f"Bank: {session.bank}")
    lines.append(f"State: {session.state_name}")
    lines.append(f"Routing: {session.routing_number}")
    lines.append(f"Account type: {session.account_type}")
    lines.append(f"Account holder: {session.account_holder_name}")
    lines.append(f"Account number: {session.masked_account_number}")
    lines.append(f"Amount: {session.amount}")
    lines.append(f"Outcome code: {session.outcome_code}")
    lines.append("")
    lines.append("Conversation history:")
    for turn in session.history:
        lines.append(f"[{turn.timestamp_utc}] {turn.speaker.upper()} ({turn.state}): {turn.message}")
    return "\\n".join(lines)


def _to_json_safe(obj: Any):
    if isinstance(obj, dict):
        return {str(k): _to_json_safe(v) for k, v in obj.items()}

    if isinstance(obj, list):
        return [_to_json_safe(v) for v in obj]

    if isinstance(obj, tuple):
        return [_to_json_safe(v) for v in obj]

    # Tipos de pandas / numpy
    if hasattr(obj, "item") and callable(obj.item):
        try:
            return obj.item()
        except Exception:
            pass

    # Datetimes y otros objetos
    if isinstance(obj, Path):
        return str(obj)

    return obj


def save_session_artifacts(session: SessionState) -> Dict[str, Path]:
    root = find_project_root()
    out_dir = root / "outputs" / "ejercicio_4" / "transcripts"
    out_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    base_name = f"ach_bot_{session.scenario}_{timestamp}"

    json_path = out_dir / f"{base_name}.json"
    txt_path = out_dir / f"{base_name}.txt"

    session_dict = _to_json_safe(session.to_dict())

    json_path.write_text(
        json.dumps(session_dict, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    txt_path.write_text(build_transcript_text(session), encoding="utf-8")

    return {"json": json_path, "txt": txt_path}
""".strip()

target_path = SRC_DIR / "ach_services.py"
target_path.write_text(ach_services_code_fixed + "\n", encoding="utf-8")

print("Archivo corregido:")
print(target_path)

Archivo corregido:
/Users/santiago.os/Documents/prueba_insights/src/ach_services.py


In [3]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

# ============================================================
# EJERCICIO 4
# PRESENTACION FINAL DE RESULTADOS DEL BOT
# VERSION SIN DEPENDER DE TABULATE
# ============================================================

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    required_dirs = {"data", "docs", "notebooks", "outputs", "src"}

    for candidate in candidates:
        if candidate.exists():
            dir_names = {p.name for p in candidate.iterdir() if p.is_dir()}
            if required_dirs.issubset(dir_names):
                return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Ejecuta este notebook dentro de ~/Documents/prueba_insights"
    )

def df_to_markdown_table(df: pd.DataFrame) -> str:
    """
    Convierte un DataFrame a una tabla markdown simple
    sin depender del paquete tabulate.
    """
    if df.empty:
        return "_Sin datos_"

    df_fmt = df.copy()

    def clean_cell(x):
        if pd.isna(x):
            return ""
        text = str(x)
        text = text.replace("\n", " ")
        text = text.replace("|", "\\|")
        return text

    headers = [clean_cell(col) for col in df_fmt.columns]
    lines = []
    lines.append("| " + " | ".join(headers) + " |")
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")

    for _, row in df_fmt.iterrows():
        values = [clean_cell(v) for v in row.tolist()]
        lines.append("| " + " | ".join(values) + " |")

    return "\n".join(lines)

ROOT = find_project_root()
DOCS_DIR = ROOT / "docs"
TRANSCRIPTS_DIR = ROOT / "outputs" / "ejercicio_4" / "transcripts"
EJ4_OUTPUT_DIR = ROOT / "outputs" / "ejercicio_4"

DOCS_DIR.mkdir(parents=True, exist_ok=True)
EJ4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not TRANSCRIPTS_DIR.exists():
    raise FileNotFoundError(
        f"No existe el directorio de transcripts: {TRANSCRIPTS_DIR}"
    )

json_files = sorted(TRANSCRIPTS_DIR.glob("*.json"))

if not json_files:
    raise FileNotFoundError(
        "No encontré transcripts JSON. "
        "Asegúrate de haber corrido los escenarios success, r01 y r03."
    )

# ============================================================
# LEER TRANSCRIPTS
# ============================================================

records = []

for path in json_files:
    data = json.loads(path.read_text(encoding="utf-8"))

    history = data.get("history", [])
    bot_turns = sum(1 for turn in history if turn.get("speaker") == "bot")
    user_turns = sum(1 for turn in history if turn.get("speaker") == "user")

    account_number_raw = data.get("account_number")
    masked_account = f"****{str(account_number_raw)[-4:]}" if account_number_raw else None

    records.append({
        "file_name": path.name,
        "scenario": data.get("scenario"),
        "bank": data.get("bank"),
        "state_name": data.get("state_name"),
        "routing_number": data.get("routing_number"),
        "account_type": data.get("account_type"),
        "account_holder_name": data.get("account_holder_name"),
        "masked_account_number": masked_account,
        "amount": data.get("amount"),
        "outcome_code": data.get("outcome_code"),
        "outcome_message": data.get("outcome_message"),
        "current_state_end": data.get("current_state"),
        "total_turns": len(history),
        "bot_turns": bot_turns,
        "user_turns": user_turns,
    })

transcripts_summary = pd.DataFrame(records).sort_values(
    ["scenario", "file_name"]
).reset_index(drop=True)

summary_display = transcripts_summary[
    [
        "scenario",
        "bank",
        "state_name",
        "routing_number",
        "account_type",
        "masked_account_number",
        "amount",
        "outcome_code",
        "total_turns",
        "bot_turns",
        "user_turns",
    ]
].copy()

summary_display["outcome_label"] = summary_display["scenario"].map({
    "success": "Success",
    "r01": "R01 - Insufficient Funds",
    "r03": "R03 - No Account / Unable to Locate Account",
})

summary_display = summary_display[
    [
        "scenario",
        "outcome_label",
        "bank",
        "state_name",
        "routing_number",
        "account_type",
        "masked_account_number",
        "amount",
        "outcome_code",
        "total_turns",
        "bot_turns",
        "user_turns",
    ]
]

messages_display = transcripts_summary[
    ["scenario", "outcome_code", "outcome_message", "file_name"]
].copy()

# ============================================================
# MOSTRAR EN NOTEBOOK
# ============================================================

display(Markdown("# Ejercicio 4 – Resultados de prueba del bot"))
display(Markdown(
    "A continuación se muestran los transcripts generados en los escenarios "
    "de prueba del prototipo CLI."
))

display(summary_display)

display(Markdown("## Mensajes finales por escenario"))
display(messages_display)

# ============================================================
# EXPORTACION DE RESULTADOS
# ============================================================

summary_display.to_csv(EJ4_OUTPUT_DIR / "ejercicio_4_bot_test_summary.csv", index=False)
messages_display.to_csv(EJ4_OUTPUT_DIR / "ejercicio_4_bot_outcomes.csv", index=False)

summary_md_table = df_to_markdown_table(summary_display)
messages_md_table = df_to_markdown_table(messages_display)

markdown_export = f"""# Ejercicio 4 - Resultados finales del prototipo ACH Funding Bot
**Prueba técnica Insights**  
**Santiago Osorio Gómez**

## Resultado general
Se ejecutaron satisfactoriamente los tres escenarios mínimos requeridos:

- `success`
- `R01`
- `R03`

## Resumen de pruebas
{summary_md_table}

## Mensajes finales por escenario
{messages_md_table}

## Lectura de resultados
El prototipo mostró un comportamiento consistente en los tres escenarios evaluados.

- En **success**, el bot completó el flujo y devolvió una confirmación de solicitud ACH creada.
- En **R01**, el bot simuló correctamente un retorno por fondos insuficientes.
- En **R03**, el bot simuló correctamente un retorno por imposibilidad de localizar la cuenta.

## Conclusión
El agente CLI implementado cumple con el alcance esperado del ejercicio:
- recolecta datos bancarios básicos;
- realiza lookup de routing number;
- explica el proceso ACH;
- confirma la información antes de enviar;
- simula outcomes relevantes;
- y conserva evidencia del flujo conversacional mediante transcripts.
"""

with open(DOCS_DIR / "ejercicio_4_resultados_finales.md", "w", encoding="utf-8") as f:
    f.write(markdown_export)

with open(EJ4_OUTPUT_DIR / "ejercicio_4_resultados_finales.txt", "w", encoding="utf-8") as f:
    f.write(markdown_export)

print("Archivos finales generados:")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_bot_test_summary.csv'}")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_bot_outcomes.csv'}")
print(f"- {DOCS_DIR / 'ejercicio_4_resultados_finales.md'}")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_resultados_finales.txt'}")

print("\\nTranscripts detectados:")
for path in json_files:
    print(f"- {path}")

# Ejercicio 4 – Resultados de prueba del bot

A continuación se muestran los transcripts generados en los escenarios de prueba del prototipo CLI.

,scenario,outcome_label,bank,state_name,routing_number,account_type,masked_account_number,amount,outcome_code,total_turns,bot_turns,user_turns
0,r01,R01 - Insufficient Funds,Bank of America,Texas,111000025,checking,****6789,2500.0,R01,18,11,7
1,r03,R03 - No Account / Unable to Locate Account,Bank of America,Texas,111000025,checking,****6789,2500.0,R03,18,11,7
2,success,Success,Bank of America,Texas,111000025,checking,****6789,2500.0,NaN,18,11,7


## Mensajes finales por escenario

,scenario,outcome_code,outcome_message,file_name
0,r01,R01,La operación fue retornada con código R01 (ins...,ach_bot_r01_20260312_222818.json
1,r03,R03,La operación fue retornada con código R03 (no ...,ach_bot_r03_20260312_222903.json
2,success,NaN,Tu solicitud ACH fue creada correctamente por ...,ach_bot_success_20260312_222724.json


Archivos finales generados:
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/ejercicio_4_bot_test_summary.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/ejercicio_4_bot_outcomes.csv
- /Users/santiago.os/Documents/prueba_insights/docs/ejercicio_4_resultados_finales.md
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/ejercicio_4_resultados_finales.txt
\nTranscripts detectados:
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/transcripts/ach_bot_r01_20260312_222818.json
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/transcripts/ach_bot_r03_20260312_222903.json
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/transcripts/ach_bot_success_20260312_222724.json


In [2]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

# ============================================================
# EJERCICIO 4
# PRESENTACION FINAL DE RESULTADOS DEL BOT
# VERSION SIN DEPENDER DE TABULATE
# ============================================================

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    required_dirs = {"data", "docs", "notebooks", "outputs", "src"}

    for candidate in candidates:
        if candidate.exists():
            dir_names = {p.name for p in candidate.iterdir() if p.is_dir()}
            if required_dirs.issubset(dir_names):
                return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Ejecuta este notebook dentro de ~/Documents/prueba_insights"
    )

def df_to_markdown_table(df: pd.DataFrame) -> str:
    """
    Convierte un DataFrame a una tabla markdown simple
    sin depender del paquete tabulate.
    """
    if df.empty:
        return "_Sin datos_"

    df_fmt = df.copy()

    def clean_cell(x):
        if pd.isna(x):
            return ""
        text = str(x)
        text = text.replace("\n", " ")
        text = text.replace("|", "\\|")
        return text

    headers = [clean_cell(col) for col in df_fmt.columns]
    lines = []
    lines.append("| " + " | ".join(headers) + " |")
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")

    for _, row in df_fmt.iterrows():
        values = [clean_cell(v) for v in row.tolist()]
        lines.append("| " + " | ".join(values) + " |")

    return "\n".join(lines)

ROOT = find_project_root()
DOCS_DIR = ROOT / "docs"
TRANSCRIPTS_DIR = ROOT / "outputs" / "ejercicio_4" / "transcripts"
EJ4_OUTPUT_DIR = ROOT / "outputs" / "ejercicio_4"

DOCS_DIR.mkdir(parents=True, exist_ok=True)
EJ4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not TRANSCRIPTS_DIR.exists():
    raise FileNotFoundError(
        f"No existe el directorio de transcripts: {TRANSCRIPTS_DIR}"
    )

json_files = sorted(TRANSCRIPTS_DIR.glob("*.json"))

if not json_files:
    raise FileNotFoundError(
        "No encontré transcripts JSON. "
        "Asegúrate de haber corrido los escenarios success, r01 y r03."
    )

# ============================================================
# LEER TRANSCRIPTS
# ============================================================

records = []

for path in json_files:
    data = json.loads(path.read_text(encoding="utf-8"))

    history = data.get("history", [])
    bot_turns = sum(1 for turn in history if turn.get("speaker") == "bot")
    user_turns = sum(1 for turn in history if turn.get("speaker") == "user")

    account_number_raw = data.get("account_number")
    masked_account = f"****{str(account_number_raw)[-4:]}" if account_number_raw else None

    records.append({
        "file_name": path.name,
        "scenario": data.get("scenario"),
        "bank": data.get("bank"),
        "state_name": data.get("state_name"),
        "routing_number": data.get("routing_number"),
        "account_type": data.get("account_type"),
        "account_holder_name": data.get("account_holder_name"),
        "masked_account_number": masked_account,
        "amount": data.get("amount"),
        "outcome_code": data.get("outcome_code"),
        "outcome_message": data.get("outcome_message"),
        "current_state_end": data.get("current_state"),
        "total_turns": len(history),
        "bot_turns": bot_turns,
        "user_turns": user_turns,
    })

transcripts_summary = pd.DataFrame(records).sort_values(
    ["scenario", "file_name"]
).reset_index(drop=True)

summary_display = transcripts_summary[
    [
        "scenario",
        "bank",
        "state_name",
        "routing_number",
        "account_type",
        "masked_account_number",
        "amount",
        "outcome_code",
        "total_turns",
        "bot_turns",
        "user_turns",
    ]
].copy()

summary_display["outcome_label"] = summary_display["scenario"].map({
    "success": "Success",
    "r01": "R01 - Insufficient Funds",
    "r03": "R03 - No Account / Unable to Locate Account",
})

summary_display = summary_display[
    [
        "scenario",
        "outcome_label",
        "bank",
        "state_name",
        "routing_number",
        "account_type",
        "masked_account_number",
        "amount",
        "outcome_code",
        "total_turns",
        "bot_turns",
        "user_turns",
    ]
]

messages_display = transcripts_summary[
    ["scenario", "outcome_code", "outcome_message", "file_name"]
].copy()

# ============================================================
# MOSTRAR EN NOTEBOOK
# ============================================================

display(Markdown("# Ejercicio 4 – Resultados de prueba del bot"))
display(Markdown(
    "A continuación se muestran los transcripts generados en los escenarios "
    "de prueba del prototipo CLI."
))

display(summary_display)

display(Markdown("## Mensajes finales por escenario"))
display(messages_display)

# ============================================================
# EXPORTACION DE RESULTADOS
# ============================================================

summary_display.to_csv(EJ4_OUTPUT_DIR / "ejercicio_4_bot_test_summary.csv", index=False)
messages_display.to_csv(EJ4_OUTPUT_DIR / "ejercicio_4_bot_outcomes.csv", index=False)

summary_md_table = df_to_markdown_table(summary_display)
messages_md_table = df_to_markdown_table(messages_display)

markdown_export = f"""# Ejercicio 4 - Resultados finales del prototipo ACH Funding Bot
**Prueba técnica Insights**  
**Santiago Osorio Gómez**

## Resultado general
Se ejecutaron satisfactoriamente los tres escenarios mínimos requeridos:

- `success`
- `R01`
- `R03`

## Resumen de pruebas
{summary_md_table}

## Mensajes finales por escenario
{messages_md_table}

## Lectura de resultados
El prototipo mostró un comportamiento consistente en los tres escenarios evaluados.

- En **success**, el bot completó el flujo y devolvió una confirmación de solicitud ACH creada.
- En **R01**, el bot simuló correctamente un retorno por fondos insuficientes.
- En **R03**, el bot simuló correctamente un retorno por imposibilidad de localizar la cuenta.

## Conclusión
El agente CLI implementado cumple con el alcance esperado del ejercicio:
- recolecta datos bancarios básicos;
- realiza lookup de routing number;
- explica el proceso ACH;
- confirma la información antes de enviar;
- simula outcomes relevantes;
- y conserva evidencia del flujo conversacional mediante transcripts.
"""

with open(DOCS_DIR / "ejercicio_4_resultados_finales.md", "w", encoding="utf-8") as f:
    f.write(markdown_export)

with open(EJ4_OUTPUT_DIR / "ejercicio_4_resultados_finales.txt", "w", encoding="utf-8") as f:
    f.write(markdown_export)

print("Archivos finales generados:")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_bot_test_summary.csv'}")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_bot_outcomes.csv'}")
print(f"- {DOCS_DIR / 'ejercicio_4_resultados_finales.md'}")
print(f"- {EJ4_OUTPUT_DIR / 'ejercicio_4_resultados_finales.txt'}")

print("\\nTranscripts detectados:")
for path in json_files:
    print(f"- {path}")

# Ejercicio 4 – Resultados de prueba del bot

A continuación se muestran los transcripts generados en los escenarios de prueba del prototipo CLI.

,scenario,outcome_label,bank,state_name,routing_number,account_type,masked_account_number,amount,outcome_code,total_turns,bot_turns,user_turns
0,r01,R01 - Insufficient Funds,Bank of America,Texas,111000025,checking,****6789,2500.0,R01,18,11,7
1,r03,R03 - No Account / Unable to Locate Account,Bank of America,Texas,111000025,checking,****6789,2500.0,R03,18,11,7
2,success,Success,Bank of America,Texas,111000025,checking,****6789,2500.0,NaN,18,11,7


## Mensajes finales por escenario

,scenario,outcome_code,outcome_message,file_name
0,r01,R01,La operación fue retornada con código R01 (ins...,ach_bot_r01_20260312_222818.json
1,r03,R03,La operación fue retornada con código R03 (no ...,ach_bot_r03_20260312_222903.json
2,success,NaN,Tu solicitud ACH fue creada correctamente por ...,ach_bot_success_20260312_222724.json


Archivos finales generados:
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/ejercicio_4_bot_test_summary.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/ejercicio_4_bot_outcomes.csv
- /Users/santiago.os/Documents/prueba_insights/docs/ejercicio_4_resultados_finales.md
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/ejercicio_4_resultados_finales.txt
\nTranscripts detectados:
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/transcripts/ach_bot_r01_20260312_222818.json
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/transcripts/ach_bot_r03_20260312_222903.json
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_4/transcripts/ach_bot_success_20260312_222724.json


## Ejercicio 4 – Resultado final del prototipo ACH Funding Bot

Se implementó un prototipo funcional de chatbot en **Python CLI**, diseñado para asistir al usuario en el proceso de fondeo de una cuenta de inversión vía **ACH**.

### Objetivo del prototipo
El agente fue construido para resolver un flujo simple y realista de funding ACH, manteniendo una arquitectura mínima, rápida de implementar y fácil de probar.

### Arquitectura aplicada
El diseño se estructuró en cuatro capas:

- **estado conversacional** (`ach_state.py`)
- **servicios auxiliares** (`ach_services.py`)
- **orquestación del flujo** (`ach_bot.py`)
- **lookup local de routing numbers** a partir de un archivo seed construido en el punto 4.1

### Flujo conversacional implementado
El bot sigue una secuencia determinística:

1. pregunta por banco;
2. pregunta por estado;
3. hace lookup del routing number;
4. explica el proceso ACH;
5. solicita tipo de cuenta;
6. solicita nombre del titular;
7. solicita account number;
8. solicita monto;
9. confirma los datos;
10. simula el resultado final.

### Casos cubiertos
El prototipo soporta tres resultados reproducibles:

- **success**
- **R01** – insufficient funds
- **R03** – no account / unable to locate account

### Criterio técnico
La lógica crítica quedó implementada completamente en Python, sin depender de un LLM para decisiones sensibles.  
Esto permite que el prototipo sea:

- más fácil de probar;
- más fácil de depurar;
- y más defendible como entregable técnico.

### Resultado
Se ejecutaron satisfactoriamente tres pruebas, una por escenario, y el sistema generó transcripts en archivos `.json` y `.txt`, dejando evidencia reproducible del comportamiento conversacional del bot.

### Conclusión
El prototipo cumple el objetivo del ejercicio: entregar un agente funcional, simple, modular y profesional, capaz de guiar un flujo ACH básico, mantener historial conversacional y simular outcomes relevantes sin sobreingeniería.